# roadstyle — example maps

This notebook builds the example maps from a small bundled dataset of real OSM road edges (Sundbyberg, Sweden). Run it top-to-bottom (Kernel → Restart & Run All) to regenerate the HTML maps in `notebooks/output/`.

**What it shows**
1. The canonical input (`RoadEdges` / `normalize_edges`)
2. Classic OSM styling (palettes & themes)
3. Filtering by road type
4. Palettes as editable JSON
5. The styler engine (class / categorical / numeric)

> Maps are written to disk with `.save()` (not shown inline) to keep this notebook small.

## 0. Setup
Imports and an `output/` folder for the rendered maps.

In [1]:
from pathlib import Path
import geopandas as gpd
import roadstyle as rs

print('roadstyle', rs.__version__)

DATA = Path('data/sundbyberg_edges.gpkg')
OUT = Path('output'); OUT.mkdir(exist_ok=True)
print('data exists:', DATA.exists())

roadstyle 0.2.0.dev0
data exists: True


## 1. The canonical input — `RoadEdges`

roadstyle's input contract is a set of road edges normalised to **EPSG:4326** with **line geometry** and a **class column** (default `highway`). You load your data however you like (here a GeoPackage); `normalize_edges()` converts any GeoDataFrame into that canonical shape (reprojecting, dropping non-line geometry, optionally renaming your class column). `render_edges()` also accepts a plain GeoDataFrame and coerces it for you.

In [2]:
raw = gpd.read_file(DATA)
print('raw  :', len(raw), 'edges | CRS', raw.crs.to_epsg(), '| columns', list(raw.columns))

edges = rs.normalize_edges(raw, class_col='highway')
print('canon:', len(edges), 'edges | CRS', edges.gdf.crs.to_epsg(),
      '| class_col', repr(edges.class_col))
print('road types:', sorted(edges.gdf['highway'].dropna().unique().tolist()))

raw  : 4039 edges | CRS 4326 | columns ['highway', 'name', 'oneway', 'maxspeed', 'geometry']
canon: 4039 edges | CRS 4326 | class_col 'highway'
road types: ['busway', 'living_street', 'primary_link', 'residential', 'secondary', 'secondary_link', 'service', 'tertiary', 'tertiary_link', 'unclassified']


## 2. Classic OSM styling — palettes & themes

`render_edges()` returns a `folium.Map`. The **highsat** palette is the high-contrast theme (cyan motorway, pink trunk, …); **carto** mimics the classic OSM look. Themes (`light`/`dark`/`satellite`) swap the road casing and default base map.

In [3]:
m1 = rs.render_edges(edges, palette='highsat', theme='dark')
m1.save(str(OUT / 'highsat_dark.html'))

m2 = rs.render_edges(edges, palette='carto', theme='light')
m2.save(str(OUT / 'carto_light.html'))

print('saved:', [p.name for p in sorted(OUT.glob('*.html'))])

saved: ['carto_light.html', 'highsat_dark.html']


## 3. Filter by road type

`include`/`exclude` keep or drop road classes before rendering. `match_links=True` means `primary` also matches `primary_link`.

In [4]:
major = rs.render_edges(edges, theme='satellite', basemap='satellite',
                        include=['motorway', 'trunk', 'primary', 'secondary'])
major.save(str(OUT / 'major_satellite.html'))
print('major-roads map saved')

major-roads map saved


## 4. Palettes as editable JSON

Built-in palettes are typed Python (the tested defaults), but any palette can be exported to **JSON**, hand-edited, and reloaded — one colour source that Python *and* a web frontend can share. `load_palette()` auto-registers by name so `render_edges(palette=name)` can use it.

In [5]:
# export -> edit -> reload
rs.save_palette('highsat', str(OUT / 'highsat.json'))

import json
doc = json.load(open(OUT / 'highsat.json'))
doc['name'] = 'my_custom'
doc['roads']['motorway']['fill'] = '#FF0000'   # motorway -> red
json.dump(doc, open(OUT / 'my_custom.json', 'w'), indent=2)

rs.load_palette(OUT / 'my_custom.json')          # registers 'my_custom'
custom = rs.render_edges(edges, palette='my_custom', theme='dark')
custom.save(str(OUT / 'custom_palette.html'))
print('custom palette registered:', 'my_custom' in rs.PALETTES)

custom palette registered: True


## 5. The styler engine

Under the hood every styling mode resolves a whole GeoDataFrame to per-edge colours/widths via a **Styler**. `ClassStyler` is the OSM class styler; `CategoricalStyler` colours by a discrete column (e.g. congestion levels); `NumericStyler` colours by a numeric column (e.g. traffic volume) with a continuous ramp. Here we inspect the resolved colours directly.

In [6]:
from collections import Counter
rf = rs.ClassStyler().resolve_frame(edges.gdf, theme='dark')
print('class styler — colour counts (top 6):')
for hexc, n in Counter(rf.fill).most_common(6):
    print(f'  {hexc}  {n:5d} edges')

class styler — colour counts (top 6):
  #FFFFFF   1901 edges
  #A6A6A6   1412 edges
  #00E676    467 edges
  #DDDDDD    183 edges
  #FFEA00     65 edges
  #FF9100     11 edges


In [7]:
# Categorical + numeric on a tiny synthetic sample (no extra data needed)
from shapely.geometry import LineString
sample = gpd.GeoDataFrame(
    {'congestion': ['low', 'moderate', 'heavy', 'severe', None],
     'aadt':       [200,    4000,       12000,   25000,    None]},
    geometry=[LineString([(i, 0), (i + 1, 1)]) for i in range(5)], crs=4326)

cat = rs.CategoricalStyler(column='congestion',
        colors={'low': '#11D68F', 'moderate': '#FFCF43',
                'heavy': '#F24E42', 'severe': '#A92727'}).resolve_frame(sample)
print('categorical fills:', cat.fill)

num = rs.NumericStyler(column='aadt', cmap='viridis',
        vmin=0, vmax=25000, width_by=(1, 8)).resolve_frame(sample)
print('numeric fills    :', num.fill)
print('numeric widths   :', [round(w, 2) for w in num.width])
print('legend           :', num.legend['kind'], num.legend['title'],
      num.legend['vmin'], '->', num.legend['vmax'])

categorical fills: ['#11D68F', '#FFCF43', '#F24E42', '#A92727', '#cccccc']
numeric fills    : ['#440457', '#453781', '#228b8d', '#fde725', '#cccccc']
numeric widths   : [1.0, 2.07, 4.33, 8.0, 1]
legend           : continuous aadt 0 -> 25000


## Done

All maps are in `notebooks/output/`. Open any `.html` in a browser for the interactive map (pan/zoom, hover-highlight, road-type filter panel, base-layer switcher).

_Data-driven maps (`render_edges(color_by=...)`) render directly once the backend wiring (Phase 3) lands; the engine above already computes the colours._